# Operations, Monitoring & Evidence 
## Stage 4 · serve → containerise → automate → monitor → retrain → govern  <font color="red">[25 marks]</font>

- Each stage lists its **sub-tasks as a checklist**
- Implement reusable logic in `app.py`, `src/monitoring.py` and `src/retrain.py`.

**File ownership** —
- *Provided:* `src/generate_current_batch.py`, `src/evaluate.py`, `tests/`, `Dockerfile`, `.github/workflows/ci.yml`, `requirements.txt`
- *You create/extend:* `app.py`, `src/monitoring.py`, `src/retrain.py` and the cells below. 
- Run after the Stage-3 notebook.

## Stage 4.1 — API Deployment <font color="red">[5 marks]</font>

- **4.1.1 — FastAPI inference service (/predict + /health + validation) [3]** *(graded from `app.py`)* — POST `/predict` (probability + label) with request validation + missing-field handling, and GET `/health`.
- **4.1.2 — Live request/response demonstrated in-notebook [2]** *(graded from this notebook)* — call `/health` and `/predict` via `fastapi.testclient.TestClient` and show the JSON.

In [1]:
# TODO 4.1.1 (app.py): FastAPI with GET /health + POST /predict, loading the model from
#   artifacts/registry, with request validation + missing-field handling.
# TODO 4.1.2 (here): use TestClient to call /health and /predict; show the responses.

from fastapi.testclient import TestClient
from app import app


# Create TestClient
client = TestClient(app)


# =========================================================
# GET /health
# =========================================================

health_response = client.get("/health")

print("GET /health")
print("Status Code:", health_response.status_code)
print("Response JSON:")
print(health_response.json())


# =========================================================
# POST /predict
# =========================================================

sample_request = {
    "features": {
        "race": "Caucasian",
        "gender": "Female",
        "age": "[70-80)",
        "admission_type_id": 1,
        "discharge_disposition_id": 1,
        "admission_source_id": 7,
        "time_in_hospital": 3,
        "medical_specialty": "InternalMedicine",
        "num_lab_procedures": 40,
        "num_procedures": 0,
        "num_medications": 10,
        "number_outpatient": 0,
        "number_emergency": 0,
        "number_inpatient": 1,
        "diag_1": "250.83",
        "diag_2": "276",
        "diag_3": "401",
        "number_diagnoses": 9,
        "max_glu_serum": "None",
        "A1Cresult": "None",
        "metformin": "No",
        "repaglinide": "No",
        "nateglinide": "No",
        "chlorpropamide": "No",
        "glimepiride": "No",
        "glipizide": "No",
        "glyburide": "No",
        "pioglitazone": "No",
        "rosiglitazone": "No",
        "acarbose": "No",
        "miglitol": "No",
        "tolazamide": "No",
        "insulin": "Steady",
        "glyburide-metformin": "No",
        "glipizide-metformin": "No",
        "change": "No",
        "diabetesMed": "Yes",
        "medical_specialty_grouped": "InternalMedicine"
    }
}


prediction_response = client.post(
    "/predict",
    json=sample_request
)


print("\nPOST /predict")
print("Status Code:", prediction_response.status_code)
print("Response JSON:")
print(prediction_response.json())

c:\Users\myacc\Downloads\Capstone MLOps 1 - Starter\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


GET /health
Status Code: 200
Response JSON:
{'status': 'healthy', 'model_loaded': True}

POST /predict
Status Code: 200
Response JSON:
{'readmission_probability': 0.0001, 'readmitted_30d': 0, 'threshold': 0.5}


## Stage 4.2 — Containerisation <font color="red">[4 marks]</font>

- **4.2.1 — Valid Dockerfile [2]** — `FROM python`, installs `libgomp1` (XGBoost), `pip install -r requirements.txt`, `CMD uvicorn`.
- **4.2.2 — Build/run evidence [2]** — a `docker build` log excerpt/screenshot + a sample `curl` response from the running container.

In [ ]:
# TODO 4.2.1: print your Dockerfile (Path('Dockerfile').read_text()).

from pathlib import Path
dockerfile_path = Path("Dockerfile")
print(dockerfile_path.read_text())

# TODO 4.2.2: include build/run evidence (build screenshot/log + sample curl response).
#       (Remember libgomp1 is required for XGBoost in the image.)


# Hospital Readmission Prediction â€” inference service
FROM python:3.11-slim

# libgomp1 is required by xgboost at runtime
# RUN apt-get update && apt-get install -y --no-install-recommends libgomp1 \
#     && rm -rf /var/lib/apt/lists/*

RUN apt-get update && \
    apt-get install -y --no-install-recommends libgomp1 && \
    rm -rf /var/lib/apt/lists/*

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py config.py ./
COPY src ./src
COPY artifacts ./artifacts

EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]



## Stage 4.3 — CI/CD Automation <font color="red">[5 marks]</font>

- **4.3.1 — GitHub Actions workflow (install deps + run pytest) [3]** — installs dependencies and runs the test suite on push/PR (ideally also validates the Docker build).
- **4.3.2 — Passing-test evidence [2]** — pytest output run here AND/OR a screenshot of a green Actions run.

In [4]:
# TODO 4.3.1: print your .github/workflows/ci.yml.

from pathlib import Path
print(Path(".github/workflows/ci.yml").read_text())

# TODO 4.3.2: run the test suite here (subprocess pytest) AND include a green-run screenshot.
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-v"],
    capture_output=True,
    text=True
)
print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

print("\nPytest exit code:", result.returncode)

name: ci

on:
  push:
    branches: [ main ]
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt
      - name: Run tests
        run: pytest tests/ -q
      - name: Validate Docker build
        run: docker build -t readmission-api:ci .

============================= test session starts =============================
platform win32 -- Python 3.14.3, pytest-9.1.1, pluggy-1.6.0 -- c:\Users\myacc\Downloads\Capstone MLOps 1 - Starter\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\myacc\Downloads\Capstone MLOps 1 - Starter
plugins: anyio-4.14.2
collecting ... collected 3 items

tests/test_app.py::test_health PASSED                                    [ 33%]
tests/test_app.py::test_predict_returns_probabi

## Stage 4.4 — Monitoring & Drift Detection <font color="red">[4 marks]</font>

Implement `run_monitoring()` in `src/monitoring.py`.

- **4.4.1 — Feature drift (Evidently, reference vs current) [2]** — run an Evidently `DataDriftPreset` (reference vs current batch); summarise + visualise.
- **4.4.2 — Prediction drift (PSI on scores) + interpretation [2]** — compute PSI on model scores and interpret (prediction drift can exceed feature-level drift).

In [ ]:
# TODO 4.4.1: generate a current batch (src.generate_current_batch); run Evidently feature drift.
# TODO 4.4.2: compute prediction PSI on model scores; summarise, plot and interpret.


## Stage 4.5 — Retraining Workflow <font color="red">[3 marks]</font>

Implement the trigger + workflow in `src/retrain.py`.

- **4.5.1 — Multi-signal retraining trigger defined [1]** — prediction PSI > 0.2 OR drifted share > 0.3 OR dataset drift.
- **4.5.2 — Retraining workflow runs + re-registers + decision record [2]** — on a trigger, retrain + register a new version and record the decision (`retraining_decision.json`).

In [ ]:
# TODO 4.5.1: define the multi-signal trigger.
# TODO 4.5.2: on a trigger, retrain + register a new version; show the decision record.


## Stage 4.6 — Logging & Governance <font color="red">[4 marks]</font>

- **4.6.1 — Prediction logging [2]** — show the `predictions.log` tail (proof predictions are logged).
- **4.6.2 — Model governance (registry history + alias + pinned deps) [2]** — show the MLflow registry version history + the `production` alias + pinned `requirements.txt`.

In [ ]:
# TODO 4.6.1: show the predictions.log tail.
# TODO 4.6.2: show the MLflow registry version history + production alias (governance).


### ✍️ Stage 4 summary
*TODO: summarise API, container/CI, monitoring, retraining and governance, and confirm all evidence is captured in this notebook + report.*